In [2]:
import pandas as pd
import numpy as np

df = pd.read_excel(r"C:\Users\Omkal\Downloads\Real_Estate_Property_Unclean_Data_500_Rows.xlsx")

In [3]:
df

,Property_ID,Property_Name,City,State,Price,Area,Bedrooms,Bathrooms,Owner_Name,Contact_Number,Listing_Date,Property_Type
0,297,Sky Heights,NaN,delhi,"-61,33,000",NaN,2,3.0,NaN,NaN,2023-07-10,Villa
1,275,City Homes,Bangalore,Karnataka,"-95,54,000",2644 sqft,4,2.0,Rahul,99abc12345,2023/01/18,Villa
2,252,Green Villa,NaN,MH,3 Cr,NaN,NaN,NaN,Neha,NaN,2024-03-24,Flat
3,118,Green Villa,NaN,NaN,"84,44,000",844 sq ft,NaN,3.0,Neha,9876543210,2023/02/24,Flat
4,251,Sky Heights,Bangalore,Karnataka,NaN,1285 sqft,two,1.0,Amit,NaN,2022-02-11,Flat
...,...,...,...,...,...,...,...,...,...,...,...,...
495,160,NaN,Delhi,Delhi,NaN,NaN,3BHK,NaN,Amit,9876543210,2023-08-10,Apartment
496,166,City Homes,Nagpur,MH,"₹46,15,000",2971 sq ft,3BHK,NaN,Priya,99abc12345,September 17 2023,Villa
497,136,City Homes,Kolkata,WB,"₹56,92,000",1842,3,3.0,Amit,99abc12345,2023/08/29,Villa
498,217,NaN,NaN,delhi,"71,17,000",2706 sqm,3BHK,2.0,Amit,12345,February 18 2022,Flat


In [4]:
df.shape
df.info()
df.isna().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Property_ID     500 non-null    int64  
 1   Property_Name   376 non-null    object 
 2   City            252 non-null    object 
 3   State           321 non-null    object 
 4   Price           407 non-null    object 
 5   Area            392 non-null    object 
 6   Bedrooms        432 non-null    object 
 7   Bathrooms       396 non-null    float64
 8   Owner_Name      404 non-null    object 
 9   Contact_Number  367 non-null    object 
 10  Listing_Date    500 non-null    object 
 11  Property_Type   500 non-null    object 
dtypes: float64(1), int64(1), object(10)
memory usage: 47.0+ KB


Property_ID         0
Property_Name     124
City              248
State             179
Price              93
Area              108
Bedrooms           68
Bathrooms         104
Owner_Name         96
Contact_Number    133
Listing_Date        0
Property_Type       0
dtype: int64

In [5]:
df = df.drop_duplicates()


In [6]:
df['State'] = df['State'].str.upper().str.strip()

state_map = {
    'MH': 'Maharashtra',
    'KARNATAKA': 'Karnataka',
    'DELHI': 'Delhi',
    'TN': 'Tamil Nadu',
    'TS': 'Telangana',
    'WB': 'West Bengal'
}

df['State'] = df['State'].replace(state_map)


In [7]:
def clean_price(val):
    if pd.isna(val):
        return np.nan
    val = str(val).replace('₹','').replace(',','').strip().lower()
    
    if 'cr' in val:
        return float(val.replace('cr','')) * 10000000
    if 'l' in val:
        return float(val.replace('l','')) * 100000
    try:
        return float(val)
    except:
        return np.nan

df['Price'] = df['Price'].apply(clean_price)

# Remove negative prices
df.loc[df['Price'] < 0, 'Price'] = np.nan


In [8]:
def clean_area(val):
    if pd.isna(val):
        return np.nan
    val = str(val).lower()
    
    if 'sqm' in val:
        return float(val.replace('sqm','').strip()) * 10.764
    if 'sqft' in val or 'sq ft' in val:
        return float(val.replace('sqft','').replace('sq ft','').strip())
    try:
        return float(val)
    except:
        return np.nan

df['Area_sqft'] = df['Area'].apply(clean_area)
df.drop(columns=['Area'], inplace=True)


In [9]:
def clean_bedrooms(val):
    if pd.isna(val):
        return np.nan
    val = str(val).lower()
    
    if 'bhk' in val:
        return int(val.replace('bhk',''))
    if val == 'two':
        return 2
    try:
        return int(val)
    except:
        return np.nan

df['Bedrooms'] = df['Bedrooms'].apply(clean_bedrooms)


In [10]:
df['Contact_Number'] = df['Contact_Number'].astype(str)
df['Contact_Number'] = df['Contact_Number'].str.replace(r'\D','', regex=True)

df.loc[df['Contact_Number'].str.len() != 10, 'Contact_Number'] = np.nan


In [11]:
df['Listing_Date'] = pd.to_datetime(df['Listing_Date'], errors='coerce')


In [12]:
df = df.fillna({
    'City': df['City'].mode()[0],
    'Property_Name': 'Unknown Property',
    'Bedrooms': df['Bedrooms'].median(),
    'Bathrooms': df['Bathrooms'].median(),
    'Price': df['Price'].median(),
    'Area_sqft': df['Area_sqft'].median()
})


In [13]:
df.isna().sum()
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Property_ID     500 non-null    int64         
 1   Property_Name   500 non-null    object        
 2   City            500 non-null    object        
 3   State           321 non-null    object        
 4   Price           500 non-null    float64       
 5   Bedrooms        500 non-null    float64       
 6   Bathrooms       500 non-null    float64       
 7   Owner_Name      404 non-null    object        
 8   Contact_Number  125 non-null    object        
 9   Listing_Date    117 non-null    datetime64[ns]
 10  Property_Type   500 non-null    object        
 11  Area_sqft       500 non-null    float64       
dtypes: datetime64[ns](1), float64(4), int64(1), object(6)
memory usage: 47.0+ KB


,Property_ID,Property_Name,City,State,Price,Bedrooms,Bathrooms,Owner_Name,Contact_Number,Listing_Date,Property_Type,Area_sqft
0,297,Sky Heights,Bangalore,Delhi,8400000.0,2.0,3.0,NaN,NaN,2023-07-10,Villa,2103.0
1,275,City Homes,Bangalore,Karnataka,8400000.0,4.0,2.0,Rahul,NaN,NaT,Villa,2644.0
2,252,Green Villa,Bangalore,Maharashtra,30000000.0,2.0,2.0,Neha,NaN,2024-03-24,Flat,2103.0
3,118,Green Villa,Bangalore,NaN,8444000.0,2.0,3.0,Neha,9876543210,NaT,Flat,844.0
4,251,Sky Heights,Bangalore,Karnataka,8400000.0,2.0,1.0,Amit,NaN,2022-02-11,Flat,1285.0


In [14]:
df.to_excel("Real_Estate_Property_Cleaned_Data.xlsx", index=False)


In [15]:
df


,Property_ID,Property_Name,City,State,Price,Bedrooms,Bathrooms,Owner_Name,Contact_Number,Listing_Date,Property_Type,Area_sqft
0,297,Sky Heights,Bangalore,Delhi,8400000.0,2.0,3.0,NaN,NaN,2023-07-10,Villa,2103.000
1,275,City Homes,Bangalore,Karnataka,8400000.0,4.0,2.0,Rahul,NaN,NaT,Villa,2644.000
2,252,Green Villa,Bangalore,Maharashtra,30000000.0,2.0,2.0,Neha,NaN,2024-03-24,Flat,2103.000
3,118,Green Villa,Bangalore,NaN,8444000.0,2.0,3.0,Neha,9876543210,NaT,Flat,844.000
4,251,Sky Heights,Bangalore,Karnataka,8400000.0,2.0,1.0,Amit,NaN,2022-02-11,Flat,1285.000
...,...,...,...,...,...,...,...,...,...,...,...,...
495,160,Unknown Property,Delhi,Delhi,8400000.0,3.0,2.0,Amit,9876543210,2023-08-10,Apartment,2103.000
496,166,City Homes,Nagpur,Maharashtra,4615000.0,3.0,2.0,Priya,NaN,NaT,Villa,2971.000
497,136,City Homes,Kolkata,West Bengal,5692000.0,3.0,3.0,Amit,NaN,NaT,Villa,1842.000
498,217,Unknown Property,Bangalore,Delhi,7117000.0,3.0,2.0,Amit,NaN,NaT,Flat,29127.384


In [16]:
city_state_map = {
    'Bangalore': 'Karnataka',
    'Mumbai': 'Maharashtra',
    'Pune': 'Maharashtra',
    'Nagpur': 'Maharashtra',
    'Delhi': 'Delhi',
    'Kolkata': 'West Bengal',
    'Chennai': 'Tamil Nadu',
    'Hyderabad': 'Telangana'
}

df['State'] = df.apply(
    lambda x: city_state_map.get(x['City'], x['State'])
    if pd.isna(x['State']) or x['State'].upper() not in city_state_map.values()
    else x['State'],
    axis=1
)


In [17]:
df['Owner_Name'] = df['Owner_Name'].fillna('Unknown')


In [18]:
df['Contact_Number'] = df['Contact_Number'].fillna('Not Available')


In [19]:
median_date = df['Listing_Date'].dropna().median()
df['Listing_Date'] = df['Listing_Date'].fillna(median_date)


In [20]:
q1 = df['Area_sqft'].quantile(0.25)
q3 = df['Area_sqft'].quantile(0.75)
iqr = q3 - q1

upper_limit = q3 + 1.5 * iqr

df.loc[df['Area_sqft'] > upper_limit, 'Area_sqft'] = df['Area_sqft'].median()


In [21]:
df.isna().sum()


Property_ID       0
Property_Name     0
City              0
State             0
Price             0
Bedrooms          0
Bathrooms         0
Owner_Name        0
Contact_Number    0
Listing_Date      0
Property_Type     0
Area_sqft         0
dtype: int64

In [22]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Property_ID     500 non-null    int64         
 1   Property_Name   500 non-null    object        
 2   City            500 non-null    object        
 3   State           500 non-null    object        
 4   Price           500 non-null    float64       
 5   Bedrooms        500 non-null    float64       
 6   Bathrooms       500 non-null    float64       
 7   Owner_Name      500 non-null    object        
 8   Contact_Number  500 non-null    object        
 9   Listing_Date    500 non-null    datetime64[ns]
 10  Property_Type   500 non-null    object        
 11  Area_sqft       500 non-null    float64       
dtypes: datetime64[ns](1), float64(4), int64(1), object(6)
memory usage: 47.0+ KB


In [23]:
df


,Property_ID,Property_Name,City,State,Price,Bedrooms,Bathrooms,Owner_Name,Contact_Number,Listing_Date,Property_Type,Area_sqft
0,297,Sky Heights,Bangalore,Karnataka,8400000.0,2.0,3.0,Unknown,Not Available,2023-07-10,Villa,2103.0
1,275,City Homes,Bangalore,Karnataka,8400000.0,4.0,2.0,Rahul,Not Available,2023-05-04,Villa,2644.0
2,252,Green Villa,Bangalore,Karnataka,30000000.0,2.0,2.0,Neha,Not Available,2024-03-24,Flat,2103.0
3,118,Green Villa,Bangalore,Karnataka,8444000.0,2.0,3.0,Neha,9876543210,2023-05-04,Flat,844.0
4,251,Sky Heights,Bangalore,Karnataka,8400000.0,2.0,1.0,Amit,Not Available,2022-02-11,Flat,1285.0
...,...,...,...,...,...,...,...,...,...,...,...,...
495,160,Unknown Property,Delhi,Delhi,8400000.0,3.0,2.0,Amit,9876543210,2023-08-10,Apartment,2103.0
496,166,City Homes,Nagpur,Maharashtra,4615000.0,3.0,2.0,Priya,Not Available,2023-05-04,Villa,2971.0
497,136,City Homes,Kolkata,West Bengal,5692000.0,3.0,3.0,Amit,Not Available,2023-05-04,Villa,1842.0
498,217,Unknown Property,Bangalore,Karnataka,7117000.0,3.0,2.0,Amit,Not Available,2023-05-04,Flat,2103.0


In [24]:
df.to_excel("Real_Estate_Property_Final_Cleaned_Data.xlsx", index=False)


In [25]:
df['Price'].mean()
df['Price'].median()
df['Price'].describe()


count    5.000000e+02
mean     9.182380e+06
std      4.879539e+06
min      3.095000e+06
25%      7.591000e+06
50%      8.400000e+06
75%      9.128500e+06
max      3.000000e+07
Name: Price, dtype: float64

In [26]:
df['City'].value_counts().head()


City
Bangalore    294
Delhi         40
Pune          34
Kolkata       33
Chennai       27
Name: count, dtype: int64

In [27]:
df[['Price', 'Area_sqft']].corr()


,Price,Area_sqft
Price,1.000000,-0.026675
Area_sqft,-0.026675,1.000000


In [28]:
df.groupby('Bedrooms')['Price'].mean()


Bedrooms
1.0    9.571157e+06
2.0    9.190067e+06
3.0    8.935196e+06
4.0    9.205266e+06
Name: Price, dtype: float64

In [29]:
df.groupby('Property_Type')['Price'].agg(['count','mean'])


,count,mean
Property_Type,,
Apartment,99,9.798273e+06
Flat,109,8.815550e+06
House,83,9.656952e+06
Penthouse,83,9.185843e+06
Villa,126,8.700905e+06


In [30]:
df.isna().sum()


Property_ID       0
Property_Name     0
City              0
State             0
Price             0
Bedrooms          0
Bathrooms         0
Owner_Name        0
Contact_Number    0
Listing_Date      0
Property_Type     0
Area_sqft         0
dtype: int64

In [31]:
np.percentile(df['Price'], [1, 99])


array([ 3292810., 30000000.])

In [32]:
df['Price_per_sqft'] = df['Price'] / df['Area_sqft']
df.groupby('City')['Price_per_sqft'].mean()


City
Bangalore    5768.441882
Chennai      4487.191463
Delhi        6464.795690
Hyderabad    5431.649763
Kolkata      5274.893056
Mumbai       5143.520797
Nagpur       5029.659524
Pune         5105.524960
Name: Price_per_sqft, dtype: float64

In [33]:
df.groupby('City')['Price'].std().sort_values(ascending=False)


City
Delhi        6.449574e+06
Kolkata      5.556133e+06
Bangalore    5.111670e+06
Mumbai       5.049184e+06
Hyderabad    3.099492e+06
Nagpur       3.010568e+06
Chennai      2.757221e+06
Pune         2.644952e+06
Name: Price, dtype: float64

In [34]:
df.groupby('City')['Price'].apply(lambda x: np.std(x)/np.mean(x))


City
Bangalore    0.548577
Chennai      0.327309
Delhi        0.593240
Hyderabad    0.377900
Kolkata      0.580162
Mumbai       0.519528
Nagpur       0.351001
Pune         0.322712
Name: Price, dtype: float64

In [35]:
df['Price_per_sqft'] = df['Price'] / df['Area_sqft']
df.sort_values('Price_per_sqft').head()


,Property_ID,Property_Name,City,State,Price,Bedrooms,Bathrooms,Owner_Name,Contact_Number,Listing_Date,Property_Type,Area_sqft,Price_per_sqft
485,299,Sky Heights,Bangalore,Karnataka,3556000.0,3.0,2.0,Amit,Not Available,2023-06-25,House,2635.0,1349.525617
359,123,City Homes,Bangalore,Karnataka,3095000.0,2.0,3.0,Amit,Not Available,2023-05-04,Apartment,2103.0,1471.707085
499,253,Sky Heights,Bangalore,Karnataka,3422000.0,2.0,2.0,Unknown,Not Available,2023-05-04,Penthouse,2271.0,1506.825187
251,160,Sky Heights,Hyderabad,Telangana,3223000.0,2.0,1.0,Neha,Not Available,2023-05-04,Flat,2103.0,1532.572515
496,166,City Homes,Nagpur,Maharashtra,4615000.0,3.0,2.0,Priya,Not Available,2023-05-04,Villa,2971.0,1553.349041


In [36]:
df['Bedrooms'].value_counts(normalize=True)*100


Bedrooms
2.0    42.0
3.0    28.6
1.0    16.6
4.0    12.8
Name: proportion, dtype: float64

In [37]:
pd.crosstab(df['City'], df['Property_Type'])


Property_Type,Apartment,Flat,House,Penthouse,Villa
City,,,,,
Bangalore,59,61,44,49,81
Chennai,6,5,5,6,5
Delhi,12,9,8,2,9
Hyderabad,6,5,3,6,7
Kolkata,3,8,9,5,8
Mumbai,6,3,6,2,4
Nagpur,2,7,0,6,9
Pune,5,11,8,7,3


In [38]:
df['Year'] = pd.to_datetime(df['Listing_Date']).dt.year
df['Year'].value_counts().sort_index()


Year
2022     44
2023    433
2024     23
Name: count, dtype: int64

In [40]:
(df.isna().mean()*100).sort_values(ascending=False)


Property_ID       0.0
Property_Name     0.0
City              0.0
State             0.0
Price             0.0
Bedrooms          0.0
Bathrooms         0.0
Owner_Name        0.0
Contact_Number    0.0
Listing_Date      0.0
Property_Type     0.0
Area_sqft         0.0
Price_per_sqft    0.0
Year              0.0
dtype: float64

In [41]:
df.corr(numeric_only=True)['Price'].sort_values(ascending=False)


Price             1.000000
Price_per_sqft    0.709567
Year              0.058072
Bathrooms         0.008032
Area_sqft        -0.026675
Bedrooms         -0.029262
Property_ID      -0.050800
Name: Price, dtype: float64

In [47]:
p99 = np.percentile(df['Price'], 99)
df_filtered = df[df['Price'] <= p99]


In [48]:
df.groupby('City')['Price'].sum().sort_values(ascending=False)


City
Bangalore    2.734845e+09
Delhi        4.294010e+08
Kolkata      3.112110e+08
Pune         2.745360e+08
Chennai      2.231940e+08
Hyderabad    2.173110e+08
Nagpur       2.015160e+08
Mumbai       1.991760e+08
Name: Price, dtype: float64

In [49]:
df[['Owner_Name','Contact_Number']].isna().mean()*100


Owner_Name        0.0
Contact_Number    0.0
dtype: float64

In [50]:
city_avg = df.groupby('City')['Price'].transform('mean')
df[(df['Price'] < city_avg) & (df['Area_sqft'] > df['Area_sqft'].mean())]


,Property_ID,Property_Name,City,State,Price,Bedrooms,Bathrooms,Owner_Name,Contact_Number,Listing_Date,Property_Type,Area_sqft,Price_per_sqft,Year
0,297,Sky Heights,Bangalore,Karnataka,8400000.0,2.0,3.0,Unknown,Not Available,2023-07-10,Villa,2103.0,3994.293866,2023
1,275,City Homes,Bangalore,Karnataka,8400000.0,4.0,2.0,Rahul,Not Available,2023-05-04,Villa,2644.0,3177.004539,2023
5,155,City Homes,Bangalore,Karnataka,5447000.0,2.0,2.0,Unknown,9876543210,2023-05-04,House,2103.0,2590.109368,2023
9,214,City Homes,Bangalore,Karnataka,8400000.0,2.0,3.0,Unknown,Not Available,2023-11-22,House,2074.0,4050.144648,2023
15,173,Sky Heights,Bangalore,Karnataka,8400000.0,1.0,1.0,Unknown,9876543210,2023-05-04,Flat,2514.0,3341.288783,2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
494,199,City Homes,Bangalore,Karnataka,3348000.0,3.0,1.0,Amit,Not Available,2023-05-04,Villa,2103.0,1592.011412,2023
495,160,Unknown Property,Delhi,Delhi,8400000.0,3.0,2.0,Amit,9876543210,2023-08-10,Apartment,2103.0,3994.293866,2023
496,166,City Homes,Nagpur,Maharashtra,4615000.0,3.0,2.0,Priya,Not Available,2023-05-04,Villa,2971.0,1553.349041,2023
498,217,Unknown Property,Bangalore,Karnataka,7117000.0,3.0,2.0,Amit,Not Available,2023-05-04,Flat,2103.0,3384.213029,2023
